# Multi-Model Evaluation Summary Notebook

This notebook builds the **initial comparison table** for all 5 models:
- DeepLabV3Plus
- SAM2
- Mask R-CNN
- SegFormer
- UNet

It reads each model's `training_results.json` generated by the existing project workflows and produces a clean table:

| CNN Architecture | Train Accuracy | Validation Accuracy | Test Accuracy |
|---|---:|---:|---:|


## How to use

1. Run model training scripts first (if you have not done so):
   - `python -m backend.scripts.deeplab_v3plus_workflow train`
   - `python -m backend.scripts.sam2_workflow`
   - `python -m backend.scripts.mask_rcnn_workflow train`
   - `python -m backend.scripts.segformer_workflow train`
   - `python -m backend.scripts.unet_workflow train`
2. Then run all notebook cells.
3. The notebook will automatically show **N/A** for models that do not yet have results files.


In [ ]:
from __future__ import annotations

import json
import subprocess
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display


In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "backend").exists():
    # If notebook is launched from another directory, try to resolve project root
    candidate = PROJECT_ROOT
    for _ in range(4):
        if (candidate / "backend").exists() and (candidate / "data").exists():
            PROJECT_ROOT = candidate
            break
        candidate = candidate.parent

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
MODELS = [
    {
        "key": "deeplabv3plus",
        "name": "DeepLabV3Plus",
        "results_path": PROJECT_ROOT / "output" / "evaluation" / "deeplabv3plus" / "training_results.json",
        "train_cmd": ["python", "-m", "backend.scripts.deeplab_v3plus_workflow", "train"],
    },
    {
        "key": "sam2",
        "name": "SAM2",
        "results_path": PROJECT_ROOT / "output" / "evaluation" / "sam2" / "training_results.json",
        "train_cmd": ["python", "-m", "backend.scripts.sam2_workflow"],
    },
    {
        "key": "maskrcnn",
        "name": "Mask R-CNN",
        "results_path": PROJECT_ROOT / "output" / "evaluation" / "maskrcnn" / "training_results.json",
        "train_cmd": ["python", "-m", "backend.scripts.mask_rcnn_workflow", "train"],
    },
    {
        "key": "segformer",
        "name": "SegFormer",
        "results_path": PROJECT_ROOT / "output" / "evaluation" / "segformer" / "training_results.json",
        "train_cmd": ["python", "-m", "backend.scripts.segformer_workflow", "train"],
    },
    {
        "key": "unet",
        "name": "UNet",
        "results_path": PROJECT_ROOT / "output" / "evaluation" / "unet" / "training_results.json",
        "train_cmd": ["python", "-m", "backend.scripts.unet_workflow", "train"],
    },
]


In [ ]:
def maybe_run_training(model: dict, run_if_missing: bool = False) -> tuple[bool, str]:
    """Run training only when requested and the result file is missing."""
    result_path = model["results_path"]
    if result_path.exists():
        return False, "results already available"
    if not run_if_missing:
        return False, "results missing (skipped training)"

    print(f"\n[TRAIN] {model['name']} -> {' '.join(model['train_cmd'])}")
    completed = subprocess.run(
        model["train_cmd"],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=False,
    )
    print(completed.stdout[-4000:])
    if completed.returncode != 0:
        print(completed.stderr[-4000:])
        return True, f"training failed (code={completed.returncode})"

    return True, "training completed"


def read_training_results(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {}


def fmt_metric(value):
    if value is None:
        return "N/A"
    try:
        return f"{float(value):.4f}"
    except Exception:
        return "N/A"


In [ ]:
RUN_TRAINING_IF_MISSING = False

rows = []
logs = []
for model in MODELS:
    _, status_msg = maybe_run_training(model, run_if_missing=RUN_TRAINING_IF_MISSING)
    payload = read_training_results(model["results_path"])

    row = {
        "CNN Architecture": model["name"],
        "Train Accuracy": fmt_metric(payload.get("train_accuracy")),
        "Validation Accuracy": fmt_metric(payload.get("val_accuracy")),
        "Test Accuracy": fmt_metric(payload.get("test_accuracy")),
    }
    rows.append(row)

    logs.append(
        {
            "model": model["name"],
            "status": status_msg,
            "results_path": str(model["results_path"]),
            "file_exists": model["results_path"].exists(),
        }
    )

summary_df = pd.DataFrame(rows)
log_df = pd.DataFrame(logs)

display(Markdown("### Evaluation Table (Initial Result)"))
display(summary_df)

display(Markdown("### Result File Status"))
display(log_df)


In [ ]:
# Optional: save outputs for report integration
output_dir = PROJECT_ROOT / "output" / "evaluation"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "initial_model_accuracy_table.csv"
md_path = output_dir / "initial_model_accuracy_table.md"

summary_df.to_csv(csv_path, index=False)

markdown_table = summary_df.to_markdown(index=False)
md_path.write_text("# Initial Model Accuracy Table\n\n" + markdown_table + "\n", encoding="utf-8")

print(f"Saved CSV: {csv_path}")
print(f"Saved Markdown: {md_path}")
